# B6.1 — Test evaluation, single-class datasets: FSC-147 test and CARPK test

Pre-declared in `results/B6_0_Setup_and_Rules/B6_rule_file.json` (recorded before any test count).

| Method | Mode | Datasets |
|---|---|---|
| GPV_R4+P[MixA] (primary), GPV_K-n+P[LM0] (secondary) — frozen `candidates_v2` | single-class, **v1.1** (ladder); v1 derived from the same run | FSC-147 test (1,190), CARPK test (459) |
| B0 — SAM2.1-Small automatic masks, count = number of masks (internal reference) | — | same |

Every image is saved as one JSON (raw predictions + diagnostics), so all metrics are computed later on CPU (B6_5).
**Runtime (L4):** ~8 h FSC-147 + ~5 h CARPK (both systems) + ~1.5 h B0. **Resumable:** after a disconnect, Run all again — finished images are skipped.
Run on an **L4** runtime: Run all. Check the output of Step 1.1 (sanity check) before leaving it overnight.

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, installs (SAN + detectron2 + diffusers, protobuf last), repo
# ============================================================
import importlib, json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"
OUT_DIR = RESULTS_ROOT / "B6_1_Ours_Single"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OMNICOUNT_ROOT = PROJECT_ROOT / "datasets" / "OmniCount-191"
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"
DATASETS = PROJECT_ROOT / "datasets"
FSC147_RAW = DATASETS / "FSC147" / "raw"
CARPK_ZIP = DATASETS / "CARPK_kaggle_coco.zip"
LOCAL = Path("/content/data"); LOCAL.mkdir(parents=True, exist_ok=True)
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| L4 expected for exact live checks")

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
%cd /content
if not Path("/content/SAN").exists():
    !git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
for p in [str(REPO_DIR), "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Repo:", REPO_COMMIT[:10])

GPU: NVIDIA L4 | L4 expected for exact live checks
/content
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.3

In [3]:
# ============================================================
# Step 0.3 — SAM2 (frozen commit), module imports, models
# ============================================================
import google.protobuf, importlib.metadata as md_meta
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session and Run all again."
import src.pipeline.b0 as b0
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()

import src.datasets as datasets_mod
import src.pipeline.semantic as semantic
import src.pipeline.prompts as prompts
import src.pipeline.candidates as C1
import src.pipeline.candidates_v2 as core

SAN_ROOT = Path("/content/SAN"); SAN_CKPT = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT.exists():
    shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT)
SAM2_CKPT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
M = core.load_models(SAN_ROOT, SAN_CKPT, SAM2_CKPT, systems=("A", "B"), device="cuda")
# ---- Frozen-code check: the pipeline files are unchanged since the B5.1 freeze
FREEZE = "2086087c5e"
FROZEN_FILES = ["src/pipeline/candidates_v2.py", "src/pipeline/candidates.py", "src/pipeline/gpv.py",
                "src/pipeline/prompts.py", "src/pipeline/semantic.py", "src/pipeline/b0.py"]
r = subprocess.run(["git", "-C", str(REPO_DIR), "diff", "--quiet", FREEZE, "HEAD", "--"] + FROZEN_FILES)
assert r.returncode == 0, "Pipeline code differs from the B5.1 freeze — stop."
RULE = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_rule_file.json"))
print("Frozen code unchanged since", FREEZE, "| rule file recorded", RULE["recorded_utc"])
# ---- B0: SAM2.1-Small, default automatic mask generator (frozen B0 setup, checkpoint SHA verified)
SAM2_S = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
amg_b0 = b0.build_amg(SAM2_S, device="cuda")
print(core.VERSION, "| models loaded:", "SAN, SAM2 predictor,", sorted(M.amg), "| normals:", M.normals is not None)
from tqdm.auto import tqdm
from PIL import Image

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
/content


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 351M/351M [00:01<00:00, 218MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


model_index.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Frozen code unchanged since 2086087c5e | rule file recorded 2026-10-26T13:32:37+00:00
candidates_v2 | models loaded: SAN, SAM2 predictor, ['LM0', 'MixA'] | normals: True


In [4]:
# ============================================================
# Step 0.4 — Test items: FSC-147 test (Drive) and CARPK test (verified Kaggle copy, extracted locally)
# ============================================================
import zipfile
fsc = datasets_mod.FSC147Dataset(FSC147_RAW, split="test", load_images=False)
ITEMS = {"fsc147": [], "carpk": []}
for i in range(len(fsc)):
    s = fsc[i]; c = next(iter(s["categories"]))
    ITEMS["fsc147"].append({"key": f"{i:05d}", "image_path": s["image_path"], "classes": [c],
                            "meta": {"dataset": "fsc147", "image_id": s["image_id"], "dataset_index": i,
                                     "gt": {c: s["categories"][c]["count"]}}})
CARPK_LOCAL = LOCAL / "carpk"
if not any(CARPK_LOCAL.rglob("test_annotations.json")):
    with zipfile.ZipFile(CARPK_ZIP) as z:
        z.extractall(CARPK_LOCAL)
cds = datasets_mod.CARPKDataset(CARPK_LOCAL, "test", load_images=False)
for i in range(len(cds)):
    s = cds[i]
    ITEMS["carpk"].append({"key": Path(s["image_path"]).stem, "image_path": s["image_path"], "classes": ["car"],
                           "meta": {"dataset": "carpk", "image_id": s["image_id"], "gt": {"car": s["categories"]["car"]["count"]}}})
assert len(ITEMS["fsc147"]) == RULE["datasets"]["FSC-147"]["images"] == 1190
assert len(ITEMS["carpk"]) == RULE["datasets"]["CARPK"]["images"] == 459
assert sum(it["meta"]["gt"]["car"] for it in ITEMS["carpk"]) == 47500
print({k: len(v) for k, v in ITEMS.items()})

{'fsc147': 1190, 'carpk': 459}


In [5]:
# ============================================================
# Step 1.1 — Sanity check before the long runs (FSC-147 val images with known B4c counts; L4 caches)
#   Live counts should equal B4c's (held-out caches were built on L4). Small GPU differences are reported.
# ============================================================
b4c = pd.read_csv(RESULTS_ROOT / "B4c_HeldOut_Candidates" / "B4c_fullval_per_image.csv")
fval = datasets_mod.FSC147Dataset(FSC147_RAW, split="val", load_images=False)
pick = b4c[(b4c["n_r4[MixA]"] > 0) & (b4c["n_r4[LM0]"] > 0)].sample(n=3, random_state=0)
rows = []
for r in pick.to_dict("records"):
    s = fval[int(r["dataset_index"])]
    assert s["image_id"] == r["image_id"]
    out = core.count_image(M, s["image_path"], [r["category"]], "single", systems=("A", "B"), ladder=True)
    m, _ = b0.run_amg(amg_b0, b0.load_image_rgb_cv2(s["image_path"]))
    rows.append({"image_id": r["image_id"], "category": r["category"], "gt": r["gt_count"],
                 "A live": out["counts"]["A"][r["category"]], "A B4c": int(r["GPV_R4+P[MixA]"]),
                 "B live": out["counts"]["B"][r["category"]], "B B4c": int(r["GPV_K-n+P[LM0]"]),
                 "B0 live": int(m.shape[0]), "B0 B4": int(r["B0"]), "t (s)": round(out["diag"]["t_total_s"], 1)})
san = pd.DataFrame(rows); display(san)
dev = max((san["A live"] - san["A B4c"]).abs().max(), (san["B live"] - san["B B4c"]).abs().max())
print("Max |live - B4c| (A, B):", dev, "| B0 reference counts were cached on another GPU — shown for information")
assert dev <= 2, "Live counts differ from B4c by more than 2 — stop and check the environment."
print("SANITY CHECK: PASS" + (" (exact)" if dev == 0 else " (small GPU differences)"))

,image_id,category,gt,A live,A B4c,B live,B B4c,B0 live,B0 B4,t (s)
0,6798.jpg,birds,87,75,75,70,70,12,12,46.7
1,1983.jpg,bottle caps,172,127,127,112,112,82,82,22.2
2,5075.jpg,chicken wings,16,20,21,16,17,12,12,20.9


Max |live - B4c| (A, B): 1 | B0 reference counts were cached on another GPU — shown for information
SANITY CHECK: PASS (small GPU differences)


In [6]:
# ============================================================
# Step 2.1 — Our two systems, single-class mode v1.1 (resumable; raw predictions per image)
#   Order: FSC-147 test, then CARPK test.
# ============================================================
for name in ["fsc147", "carpk"]:
    t0 = time.time()
    n = core.run_batch(M, ITEMS[name], OUT_DIR / name / "rows_ours", "single", systems=("A", "B"), ladder=True)
    print(f"{name}: processed now {n} | {(time.time() - t0) / 3600:.2f} h")

candidates_v2 single:   0%|          | 0/1190 [00:00<?, ?it/s]

fsc147: processed now 1190 | 7.25 h


candidates_v2 single:   0%|          | 0/459 [00:00<?, ?it/s]

carpk: processed now 459 | 3.68 h


In [7]:
# ============================================================
# Step 2.2 — B0 (SAM2.1-Small automatic masks, count = number of masks), same images (resumable)
# ============================================================
for name in ["fsc147", "carpk"]:
    rows_dir = OUT_DIR / name / "rows_b0"; rows_dir.mkdir(parents=True, exist_ok=True)
    todo = [it for it in ITEMS[name] if not (rows_dir / f"{it['key']}.json").exists()]
    for it in tqdm(todo, desc=f"B0 {name}"):
        t0 = time.time()
        m, _ = b0.run_amg(amg_b0, b0.load_image_rgb_cv2(it["image_path"]))
        rec = {"key": it["key"], "image_path": str(it["image_path"]), "classes": it["classes"], "meta": it["meta"],
               "version": "B0 (b0.py, SAM2.1-Small default AMG)", "counts": {"B0": {it["classes"][0]: int(m.shape[0])}},
               "diag": {"t_total_s": time.time() - t0}}
        tmp = rows_dir / f"{it['key']}.json.tmp"; tmp.write_text(json.dumps(rec)); tmp.replace(rows_dir / f"{it['key']}.json")
    print(name, "B0 rows:", len(list(rows_dir.glob("*.json"))))

B0 fsc147:   0%|          | 0/1190 [00:00<?, ?it/s]

fsc147 B0 rows: 1190


B0 carpk:   0%|          | 0/459 [00:00<?, ?it/s]

carpk B0 rows: 459


In [8]:
# ============================================================
# Step 3.1 — Completeness check, progress summary (full analysis: B6_5), metadata
# ============================================================
def load_rows(d):
    return {p.stem: json.load(open(p)) for p in Path(d).glob("*.json")}
summary, ladder, done = [], [], True
for name in ["fsc147", "carpk"]:
    ours, b0r = load_rows(OUT_DIR / name / "rows_ours"), load_rows(OUT_DIR / name / "rows_b0")
    keys = [it["key"] for it in ITEMS[name]]
    complete = all(k in ours for k in keys) and all(k in b0r for k in keys)
    done &= complete
    if not complete:
        print(f"{name}: incomplete — ours {len(ours)}/{len(keys)}, B0 {len(b0r)}/{len(keys)}"); continue
    gt = np.array([ours[k]["meta"]["gt"][ours[k]["classes"][0]] for k in keys], float)
    preds = {}
    for s_, lab in [("A", "GPV_R4+P[MixA]"), ("B", "GPV_K-n+P[LM0]")]:
        v11 = np.array([ours[k]["counts"][s_][ours[k]["classes"][0]] for k in keys], float)
        lvl = np.array([ours[k]["diag"][f"{s_}_ladder_level"] for k in keys])
        preds[f"{lab} v1.1"] = v11; preds[f"{lab} v1"] = np.where(lvl >= 1, 0, v11)
        ladder.append({"dataset": name, "system": lab, "ladder fired (L1/L2)": int((lvl >= 1).sum()), "count 0, all levels empty": int((lvl == -1).sum())})
    preds["B0"] = np.array([b0r[k]["counts"]["B0"][b0r[k]["classes"][0]] for k in keys], float)
    for lab, p in preds.items():
        e = p - gt
        summary.append({"dataset": name, "method": lab, "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
                        "NAE": (np.abs(e) / gt).mean(), "SRE": np.sqrt(((e ** 2) / gt).mean())})
    tt = np.mean([ours[k]["diag"]["t_total_s"] for k in keys])
    print(f"{name}: mean time per image (both systems) {tt:.1f} s")
sm = pd.DataFrame(summary)
print("Progress summary (descriptive; the pre-declared analysis runs in B6_5):"); display(sm.round(2))
print("\nLadder usage:"); display(pd.DataFrame(ladder))
sm.to_csv(OUT_DIR / "B6_1_progress_summary.csv", index=False)
with open(OUT_DIR / "B6_1_metadata.json", "w") as f:
    json.dump({"notebook": "B6_1_Ours_Single", "module": core.VERSION, "freeze_commit": FREEZE, "repo_commit": REPO_COMMIT,
               "gpu": GPU_NAME, "rule_file_recorded_utc": RULE["recorded_utc"], "complete": bool(done),
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "carpk_source": "CARPK_kaggle_coco.zip (verified)", "torch": torch.__version__}, f, indent=2)
print("COMPLETE" if done else "NOT COMPLETE — Run all again to continue", "| saved to", OUT_DIR)

fsc147: mean time per image (both systems) 21.9 s
carpk: mean time per image (both systems) 28.8 s
Progress summary (descriptive; the pre-declared analysis runs in B6_5):


,dataset,method,MAE,RMSE,NAE,SRE
0,fsc147,GPV_R4+P[MixA] v1.1,24.56,136.59,0.39,4.35
1,fsc147,GPV_R4+P[MixA] v1,24.76,136.91,0.39,4.38
2,fsc147,GPV_K-n+P[LM0] v1.1,26.87,139.57,0.35,5.24
3,fsc147,GPV_K-n+P[LM0] v1,26.88,139.60,0.35,5.25
4,fsc147,B0,33.06,144.08,0.40,4.91
5,carpk,GPV_R4+P[MixA] v1.1,7.27,9.90,0.09,0.93
6,carpk,GPV_R4+P[MixA] v1,7.27,9.90,0.09,0.93
7,carpk,GPV_K-n+P[LM0] v1.1,15.13,19.16,0.15,1.74
8,carpk,GPV_K-n+P[LM0] v1,15.13,19.16,0.15,1.74
9,carpk,B0,30.88,41.26,0.61,4.88



Ladder usage:


,dataset,system,ladder fired (L1/L2),"count 0, all levels empty"
0,fsc147,GPV_R4+P[MixA],3,0
1,fsc147,GPV_K-n+P[LM0],3,0
2,carpk,GPV_R4+P[MixA],0,0
3,carpk,GPV_K-n+P[LM0],0,0


COMPLETE | saved to /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_1_Ours_Single
